# Dosimetry Database EDA
Exploratory Data Analysis of the Matt temperature data

In [2]:
import os
import re
import glob
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple

# Configuration
ROOT_DIR = "."  # Root directory to search for .txt files
TXT_PATTERN = "**/*.txt"  # Pattern to find .txt files in subdirectories

## 1. Parse .txt Files and Extract Timestamps

Parse all .txt files in subdirectories and extract timestamps on a per-device basis.

In [1]:
def parse_txt_files(root_dir: str = ".", pattern: str = "**/*.txt") -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parse .txt log files and extract timestamps and device information.
    
    Returns:
        (sessions, readings) - DataFrames compatible with the rest of the notebook
    """
    # Find all .txt files
    txt_files = list(Path(root_dir).glob(pattern))
    # Filter out non-data files (like "Note on D560.txt")
    txt_files = [f for f in txt_files if f.is_file() and "Batch" in f.name]
    
    print(f"Found {len(txt_files)} .txt file(s) to parse...")
    
    all_readings = []
    
    for txt_file in txt_files:
        print(f"  Parsing {txt_file.name}...")
        with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        
        for line in lines:
            line = line.strip()
            
            # Match pattern: CB100-XXXXX--> TS: <timestamp>.<milliseconds> | Pulse: ... | Charge: ...
            # Example: [13:23:16.906] CB100-2599429--> TS: 1766751796.271 | Pulse:      0 | Charge:     58 | ...
            match = re.search(r'CB100-(\d+)-->.*?TS:\s*(\d+)\.(\d+).*?Pulse:\s*(\d+).*?Charge:\s*(\d+)', line)
            if match:
                device_serial = f"CB100-{match.group(1)}"
                ts_seconds = int(match.group(2))
                ts_milliseconds = int(match.group(3))
                pulse_count = int(match.group(4))
                charge_count = int(match.group(5))
                
                # Convert Unix timestamp to datetime
                # Unix timestamp is in seconds, milliseconds are fractional
                timestamp = datetime.fromtimestamp(ts_seconds + ts_milliseconds / 1000.0)
                
                all_readings.append({
                    'device_uid': device_serial,
                    'dosimeter_label': device_serial,  # Using device_uid as dosimeter_label
                    'captured_at': timestamp,
                    'created_at': timestamp,  # Same as captured_at for now
                    'charge_count': charge_count,
                    'pulse_count': pulse_count,
                    'dose': None,  # Not available in raw logs
                    'dose_mrem_per_sec': None,  # Not available in raw logs
                    'cumulative_dose_mrem': None,  # Not available in raw logs
                    'case_id': None,  # Not available in raw logs
                    'dosimeter_id': None,  # Not available in raw logs
                    'source_file': txt_file.name
                })
    
    if not all_readings:
        print("No readings found in .txt files!")
        return pd.DataFrame(), pd.DataFrame()
    
    readings_df = pd.DataFrame(all_readings)
    readings_df['captured_at'] = pd.to_datetime(readings_df['captured_at'])
    readings_df['created_at'] = pd.to_datetime(readings_df['created_at'])
    
    # Extract dosimeter number from label (last 3 digits)
    readings_df['dosimeter_num'] = readings_df['dosimeter_label'].str[-3:].astype(int)
    
    # Create sessions based on time windows
    # Group by source file and device, then create sessions
    sessions_list = []
    session_id_counter = 1
    
    for source_file in readings_df['source_file'].unique():
        file_readings = readings_df[readings_df['source_file'] == source_file]
        
        # Group by device within this file
        for device in file_readings['device_uid'].unique():
            device_readings = file_readings[file_readings['device_uid'] == device].sort_values('captured_at')
            
            if len(device_readings) > 0:
                started_at = device_readings['captured_at'].min()
                ended_at = device_readings['captured_at'].max()
                
                sessions_list.append({
                    'id': session_id_counter,
                    'started_at': started_at,
                    'ended_at': ended_at,
                    'source_file': source_file,
                    'device_uid': device
                })
                session_id_counter += 1
    
    sessions_df = pd.DataFrame(sessions_list)
    if not sessions_df.empty:
        sessions_df['started_at'] = pd.to_datetime(sessions_df['started_at'])
        sessions_df['ended_at'] = pd.to_datetime(sessions_df['ended_at'])
    
    # Assign monitoring_session_id to readings based on device and file
    readings_df['monitoring_session_id'] = None
    for _, session in sessions_df.iterrows():
        mask = (readings_df['source_file'] == session['source_file']) & \
               (readings_df['device_uid'] == session['device_uid']) & \
               (readings_df['captured_at'] >= session['started_at']) & \
               (readings_df['captured_at'] <= session['ended_at'])
        readings_df.loc[mask, 'monitoring_session_id'] = session['id']
    
    # Add id column to readings
    readings_df['id'] = range(1, len(readings_df) + 1)
    
    print(f"\nParsed {len(readings_df)} readings from {len(sessions_df)} sessions")
    print(f"Devices found: {readings_df['device_uid'].nunique()}")
    print(f"Date range: {readings_df['captured_at'].min()} to {readings_df['captured_at'].max()}")
    
    return sessions_df, readings_df

# Parse all .txt files
sessions, readings = parse_txt_files(ROOT_DIR, TXT_PATTERN)

NameError: name 'Tuple' is not defined

In [ ]:
# Summary of parsed data
print("Data Summary")
print("=" * 80)
print(f"Total sessions: {len(sessions)}")
print(f"Total readings: {len(readings)}")
print(f"Unique devices: {readings['device_uid'].nunique() if not readings.empty else 0}")
print(f"Unique dosimeters: {readings['dosimeter_num'].nunique() if not readings.empty else 0}")

if not sessions.empty:
    print(f"\nSession date range: {sessions['started_at'].min()} to {sessions['ended_at'].max()}")
    print(f"\nSessions by file:")
    print(sessions.groupby('source_file').size())

if not readings.empty:
    print(f"\nReadings by device:")
    print(readings.groupby('device_uid').size())

In [ ]:
# Display sample data
if not sessions.empty:
    print("\nSample Sessions:")
    print("=" * 80)
    display(sessions.head(10))

if not readings.empty:
    print("\nSample Readings:")
    print("=" * 80)
    display(readings[['id', 'device_uid', 'dosimeter_label', 'captured_at', 'charge_count', 'pulse_count']].head(10))

## 2. Timestamp Distribution Analysis

In [ ]:
# Analyze timestamp distribution by device
timestamp_data = {}

if not readings.empty:
    for device in readings['device_uid'].unique():
        device_readings = readings[readings['device_uid'] == device]
        if len(device_readings) > 0:
            timestamp_data[device] = device_readings['captured_at']
            print(f"{device}: {len(device_readings)} records, {device_readings['captured_at'].min()} to {device_readings['captured_at'].max()}")

In [ ]:
# Histogram of timestamps
if timestamp_data:
    n_tables = len(timestamp_data)
    fig, axes = plt.subplots(n_tables, 1, figsize=(12, 3 * n_tables))
    if n_tables == 1:
        axes = [axes]

    for ax, (table, timestamps) in zip(axes, timestamp_data.items()):
        ax.hist(timestamps, bins=30, edgecolor='black', alpha=0.7)
        ax.set_title(f'{table} - created_at distribution')
        ax.set_xlabel('Date')
        ax.set_ylabel('Count')
        ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

## 3. Monitoring Sessions by Date

Filter and plot sessions from a specific date.

In [ ]:
# Filter for specific date (change this to select different days)
FILTER_DATE = "2025-12-26"  # Updated default date based on sample files

# Filter sessions to selected date
if not sessions.empty:
    sessions_filtered = sessions[sessions['started_at'].dt.strftime('%Y-%m-%d') == FILTER_DATE].reset_index(drop=True)
else:
    sessions_filtered = pd.DataFrame()

# Filter readings to match filtered sessions
if not readings.empty and not sessions_filtered.empty:
    session_ids = sessions_filtered['id'].tolist()
    readings_filtered = readings[readings['monitoring_session_id'].isin(session_ids)].copy()
else:
    readings_filtered = pd.DataFrame()

print(f"Found {len(sessions_filtered)} sessions on {FILTER_DATE}")
if not sessions_filtered.empty:
    display(sessions_filtered)

In [ ]:
# Plot each monitoring session in its own panel
if len(sessions_filtered) == 0:
    print("No sessions found for this date")
elif readings_filtered.empty:
    print("No readings found for filtered sessions")
else:
    fig, axes = plt.subplots(len(sessions_filtered), 1, figsize=(14, 4 * len(sessions_filtered)))
    if len(sessions_filtered) == 1:
        axes = [axes]

    for idx, (_, session) in enumerate(sessions_filtered.iterrows()):
        ax = axes[idx]
        session_id = session['id']
        
        # Get readings for this session
        session_readings = readings_filtered[readings_filtered['monitoring_session_id'] == session_id]
        
        if len(session_readings) == 0:
            ax.text(0.5, 0.5, 'No readings for this session', 
                   transform=ax.transAxes, ha='center', va='center')
            continue
        
        # Plot charge_count for each dosimeter (since dose_mrem_per_sec is not available)
        for dos_num in sorted(session_readings['dosimeter_num'].unique()):
            dos_data = session_readings[session_readings['dosimeter_num'] == dos_num].sort_values('captured_at')
            ax.plot(dos_data['captured_at'], dos_data['charge_count'], 
                    label=f'DOS-{dos_num}', linewidth=0.8, marker='o', markersize=2)
        
        # Format with dynamic title
        start_str = session['started_at'].strftime('%H:%M:%S')
        end_str = session['ended_at'].strftime('%H:%M:%S')
        ax.set_title(f"Session {idx + 1} ({start_str} - {end_str} UTC)")
        ax.set_ylabel('Charge Count')
        ax.legend(loc='upper right', ncol=2)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (UTC)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Summary stats per session
print("Session Summary")
print("=" * 80)
for idx, (_, session) in enumerate(sessions_filtered.iterrows()):
    session_id = session['id']
    session_readings = readings_filtered[readings_filtered['monitoring_session_id'] == session_id]
    dosimeters = sorted(session_readings['dosimeter_num'].unique()) if not session_readings.empty else []
    
    print(f"\nSession {idx + 1}")
    print(f"  Time: {session['started_at']} to {session['ended_at']}")
    print(f"  Device: {session.get('device_uid', 'N/A')}")
    print(f"  Dosimeters: {[int(d) for d in dosimeters]}")
    print(f"  Total readings: {len(session_readings)}")

In [ ]:
# Analyze reporting frequency per session and dosimeter
print("Reporting Frequency Analysis")
print("=" * 80)

freq_data = []

for idx, (_, session) in enumerate(sessions_filtered.iterrows()):
    session_id = session['id']
    session_readings = readings_filtered[readings_filtered['monitoring_session_id'] == session_id]
    
    if session_readings.empty:
        continue
    
    print(f"\nSession {idx + 1}")
    print("-" * 60)
    
    for dos_num in sorted(session_readings['dosimeter_num'].unique()):
        dos_data = session_readings[session_readings['dosimeter_num'] == dos_num].sort_values('captured_at')
        
        # Calculate time deltas between consecutive readings
        time_diffs = dos_data['captured_at'].diff().dropna()
        
        if len(time_diffs) > 0:
            avg_ms = time_diffs.mean().total_seconds() * 1000
            std_ms = time_diffs.std().total_seconds() * 1000
            min_ms = time_diffs.min().total_seconds() * 1000
            max_ms = time_diffs.max().total_seconds() * 1000
            freq_hz = 1000 / avg_ms if avg_ms > 0 else 0
            
            print(f"  DOS-{dos_num}: avg={avg_ms:.1f}ms (±{std_ms:.1f}ms), range=[{min_ms:.1f}, {max_ms:.1f}]ms, ~{freq_hz:.1f} Hz")
            
            freq_data.append({
                'session': idx + 1,
                'dosimeter': int(dos_num),
                'avg_ms': avg_ms,
                'std_ms': std_ms,
                'freq_hz': freq_hz
            })

freq_df = pd.DataFrame(freq_data)

In [ ]:
# Visualize frequency consistency
if len(freq_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Box plot of avg frequency by session
    session_nums = sorted(freq_df['session'].unique())
    axes[0].boxplot([freq_df[freq_df['session'] == s]['freq_hz'] for s in session_nums], 
                    labels=[f'Session {s}' for s in session_nums])
    axes[0].set_ylabel('Frequency (Hz)')
    axes[0].set_title('Reporting Frequency by Session')
    axes[0].grid(True, alpha=0.3)

    # Bar chart comparing all dosimeters
    axes[1].bar(range(len(freq_df)), freq_df['freq_hz'], color='steelblue', alpha=0.7)
    axes[1].set_xticks(range(len(freq_df)))
    axes[1].set_xticklabels([f"S{r['session']}-{r['dosimeter']}" for _, r in freq_df.iterrows()], rotation=45, ha='right')
    axes[1].set_ylabel('Frequency (Hz)')
    axes[1].set_title('Reporting Frequency by Dosimeter')
    axes[1].axhline(freq_df['freq_hz'].mean(), color='red', linestyle='--', label=f"Mean: {freq_df['freq_hz'].mean():.1f} Hz")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"\nOverall: Mean={freq_df['freq_hz'].mean():.2f} Hz, Std={freq_df['freq_hz'].std():.2f} Hz")
else:
    print("No frequency data to plot")

## 4. Export Session Data to CSV

In [ ]:
os.makedirs('session_exports', exist_ok=True)

for idx, (_, session) in enumerate(sessions_filtered.iterrows()):
    session_id = session['id']
    session_readings = readings_filtered[readings_filtered['monitoring_session_id'] == session_id].copy()

    if session_readings.empty:
        continue

    # Add session start/end times to each row
    session_readings['session_started_at'] = session['started_at']
    session_readings['session_ended_at'] = session['ended_at']

    # Select columns for export (only include columns that exist)
    export_cols = [
        'id',
        'monitoring_session_id',
        'case_id',
        'dosimeter_id',
        'dosimeter_label',
        'device_uid',
        'charge_count',
        'pulse_count',
        'dose',
        'dose_mrem_per_sec',
        'cumulative_dose_mrem',
        'captured_at',
        'created_at',
        'session_started_at',
        'session_ended_at'
    ]
    # Only include columns that exist in the dataframe
    export_cols = [c for c in export_cols if c in session_readings.columns]
    export_df = session_readings[export_cols].copy()
    export_df = export_df.sort_values(['dosimeter_label', 'captured_at'])

    # Create filename with date and session number
    date_str = session['started_at'].strftime('%Y%m%d')
    start_time = session['started_at'].strftime('%H%M')
    filename = f"session_exports/{date_str}_session{idx+1}_{start_time}.csv"

    export_df.to_csv(filename, index=False)
    print(f"Exported: {filename} ({len(export_df)} readings)")

In [ ]:
# Data parsing complete - sessions and readings DataFrames are ready for analysis
print("\nData parsing complete!")
print(f"Total sessions: {len(sessions)}")
print(f"Total readings: {len(readings)}")